# Human vs NHP pipeline

This is an example notebook to analyze the expression distribution of the *IQGAP2* gene between macrophages and microglia in humans and non-human primates (macaques). The work with the other genes followed the same logic.

In [ ]:
import scanpy as sc

# Read all files
adata1 = sc.read_h5ad("Dataset_1.h5ad")
adata2 = sc.read_h5ad("Dataset_2.h5ad")
adata3 = sc.read_h5ad("Dataset_3.h5ad")
adata4 = sc.read_h5ad("Dataset_4.h5ad")
adata5 = sc.read_h5ad("Dataset_5.h5ad")
adata6 = sc.read_h5ad("Dataset_6.h5ad")
adata7 = sc.read_h5ad("Dataset_7.h5ad")
adata8 = sc.read_h5ad("Dataset_8.h5ad")
adata9 = sc.read_h5ad("Dataset_9.h5ad")

# LogFC (macrophage vs microglia) calculation

The main function calculates the gene expression for macrophages and microglia for a given **donor** (to avoid pseudoreplication).

In [ ]:
import pandas as pd
import numpy as np

def calculate_donor_logfc(adata, gene_name, macro_label, 
                          cluster_col='majority_voting', 
                          donor_col='sample'):
    """
    Calculates Log2FC for the gene between macrophages and microglia of the given donor.
    """
    if gene_name not in adata.var_names:
        raise ValueError(f"The gene {gene_name} was not found in adata.var_names")
    
    # Get gene expression
    gene_expr = adata[:, gene_name].X
    if hasattr(gene_expr, 'toarray'):
        gene_expr = gene_expr.toarray().flatten()
    else:
        gene_expr = gene_expr.flatten()
    
    results = []
    
    for donor in adata.obs[donor_col].unique():
        # Filter cells of this donor
        donor_mask = adata.obs[donor_col] == donor
        donor_expr = gene_expr[donor_mask]
        donor_clusters = adata.obs.loc[donor_mask, cluster_col]
        
        # Separate macrophages and microglia
        macro_mask = donor_clusters == macro_label
        micro_mask = ~macro_mask
        
        macro_expr = donor_expr[macro_mask]
        micro_expr = donor_expr[micro_mask]
        
        # Skip donor if macro or micro cells are missing
        if len(macro_expr) == 0 or len(micro_expr) == 0:
            continue
        
        # Log2FC = difference between means (since data are already log-normalized)
        logfc = np.mean(macro_expr) - np.mean(micro_expr)
        
        results.append({
            'donor': donor,
            'log2fc': logfc
        })
    
    return pd.DataFrame(results)
    

In [ ]:
# Run for adata1:
result_df_1 = calculate_donor_logfc(
    adata=adata1,
    gene_name='IQGAP2',
    macro_label='Macro F13A1 COLEC12',
    cluster_col='majority_voting',
    donor_col='sample'
)

result_df_1

In [ ]:
# Run for adata2:
result_df_2 = calculate_donor_logfc(
    adata=adata2,
    gene_name='IQGAP2',
    macro_label='Macro F13A1 COLEC12',
    cluster_col='majority_voting',
    donor_col='sample'
)

result_df_2

In [ ]:
# Run for adata3:
result_df_3 = calculate_donor_logfc(
    adata=adata3,
    gene_name='IQGAP2',
    macro_label='Macro F13A1 COLEC12',
    cluster_col='majority_voting',
    donor_col='sample'
)

result_df_3

In [ ]:
# Run for adata4:
result_df_4 = calculate_donor_logfc(
    adata=adata4,
    gene_name='IQGAP2',
    macro_label='Macro F13A1 COLEC12',
    cluster_col='majority_voting',
    donor_col='sample'
)

result_df_4

In [ ]:
# Run for adata5:
result_df_5 = calculate_donor_logfc(
    adata=adata5,
    gene_name='IQGAP2',  
    macro_label='Macro F13A1 COLEC12',
    cluster_col='majority_voting',
    donor_col='sample'
)

result_df_5

In [ ]:
# Run for adata8:
result_df_8 = calculate_donor_logfc(
    adata=adata8,
    gene_name='IQGAP2',  
    macro_label='Macro F13A1 COLEC12',
    cluster_col='majority_voting',
    donor_col='sample'
)

result_df_8

In [ ]:
# Run for adata9:
result_df_9 = calculate_donor_logfc(
    adata=adata9,
    gene_name='IQGAP2',  
    macro_label='Macro F13A1 COLEC12',
    cluster_col='majority_voting',
    donor_col='sample'
)

result_df_9

In [ ]:
# Run for adata6:
result_df_6 = calculate_donor_logfc(
    adata=adata6,
    gene_name='IQGAP2',  
    macro_label='PVM',
    cluster_col='majority_voting',
    donor_col='sample'
)

result_df_6

In [ ]:
# Run for adata7:
result_df_7 = calculate_donor_logfc(
    adata=adata7,
    gene_name='IQGAP2',  
    macro_label='14',
    cluster_col='leiden_res_2.0',
    donor_col='sample'
)

result_df_7

In [ ]:
len(result_df_1) + len(result_df_2) + len(result_df_3) + len(result_df_4) + len(result_df_5) + len(result_df_8) + len(result_df_9)

In [ ]:
len(result_df_6) + len(result_df_7)

After filtering out donors lacking sufficient microglial or macrophage cell numbers, we successfully calculated donor-specific logFC values for 114 human and 17 non-human primate (macaque) subjects across all processed datasets.

# Linear Mixed-Effects Model

This is a critical step of our analysis. We compare the logFC values of 114 human donors against 17 macaque donors using a **Linear Mixed-Effects Model (LMM)**.

We test whether the expression differences are significantly related to Species, while using Dataset as a random effect. Technically, the dataset effect accounts for study-specific factors, including brain regions and clinical conditions.

In [ ]:
import statsmodels.formula.api as smf

# Aggregate all processed donor data
all_data = []

# Human datasets (1-5, 8, 9)
human_datasets = [
    (result_df_1, 'DS1'),
    (result_df_2, 'DS2'),
    (result_df_3, 'DS3'),
    (result_df_4, 'DS4'),
    (result_df_5, 'DS5'),
    (result_df_8, 'DS8'),
    (result_df_9, 'DS9'),
]

for df, ds_id in human_datasets:
    df_copy = df.copy()
    df_copy['Dataset'] = ds_id
    df_copy['Species'] = 'Human'
    all_data.append(df_copy)

# Non-human primate (NHP) / Macaque datasets (6, 7)
nhp_datasets = [
    (result_df_6, 'DS6'),
    (result_df_7, 'DS7'),
]

for df, ds_id in nhp_datasets:
    df_copy = df.copy()
    df_copy['Dataset'] = ds_id
    df_copy['Species'] = 'NHP'
    all_data.append(df_copy)

# Concatenate into a single master dataframe
df_all = pd.concat(all_data, ignore_index=True)

print(f"Total number of observations: {len(df_all)}")
print(df_all.groupby('Species').size())

# Fit Linear Mixed-Effects Model (LMM) with random intercept per dataset
model = smf.mixedlm("log2fc ~ Species", data=df_all, groups=df_all["Dataset"])
result = model.fit()

print("\n" + "="*50)
print("LMM RESULTS: IQGAP2 Human vs NHP")
print("="*50)
print(result.summary())

# Extract key statistical parameters
species_coef = result.params.get('Species[T.NHP]')
species_pval = result.pvalues.get('Species[T.NHP]')

print(f"SPECIES EFFECT (NHP vs Human reference):")
print(f"Coefficient: {species_coef:.4f}")
print(f"P-value: {species_pval:.4e}")
print(f"Significant: {'YES ***' if species_pval < 0.001 else 'YES **' if species_pval < 0.01 else 'YES *' if species_pval < 0.05 else 'NO'}")

This highly significant **p-value P < 0.001** demonstrates that macaques have a significantly higher logFC than humans (**β = 1.5038**). This result confirms a strong evolutionary divergence in the microglial gene co-expression program between the two species.

# Visualization

Here we visualize the distribution of logFC values across all donors for each dataset. This plot helps to visually compare the trends between human and non-human primate datasets.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Combine all processed dataframes into a single plotting data structure
dfs = [result_df_1, result_df_2, result_df_3, result_df_4,
       result_df_5, result_df_8, result_df_9, result_df_6, result_df_7]
ids = ['DS1', 'DS2', 'DS3', 'DS4', 'DS5', 'DS8', 'DS9', 'DS6', 'DS7']

plot_df = pd.concat([df.assign(Dataset=idx) for df, idx in zip(dfs, ids)], ignore_index=True)

# 2. Define dataset labels and formatting order
ds_labels = {
    'DS1': '$\\mathbf{DS1}$\n(Human)', 'DS2': '$\\mathbf{DS2}$\n(Human)',
    'DS3': '$\\mathbf{DS3}$\n(Human)', 'DS4': '$\\mathbf{DS4}$\n(Human)',
    'DS5': '$\\mathbf{DS5}$\n(Human)', 'DS8': '$\\mathbf{DS8}$\n(Human)',
    'DS9': '$\\mathbf{DS9}$\n(Human)',
    'DS6': '$\\mathbf{DS6}$\n($\\mathit{M.fascicularis}$)',
    'DS7': '$\\mathbf{DS7}$\n($\\mathit{M.mulatta}$)'
}
plot_df['Label'] = plot_df['Dataset'].map(ds_labels)
label_order = [ds_labels[i] for i in ids]

# 3. Assign color scheme: Human = skyblue, Non-human primate (NHP) = coral
plot_df['Color'] = plot_df['Dataset'].apply(lambda x: 'coral' if x in ['DS6', 'DS7'] else 'skyblue')

# 4. Generate visualization
FONT_BIG, FONT_SMALL = 23, 18

fig, ax = plt.subplots(figsize=(10, 10), dpi=300)
sns.set_style("white")

# Draw boxplots with specified dataset-specific palette
palette = dict(zip(plot_df['Label'], plot_df['Color']))
sns.boxplot(
    data=plot_df, x='log2fc', y='Label', order=label_order,
    palette=palette, width=0.6, showfliers=False,
    linewidth=1.5, linecolor='grey', ax=ax
)

# Overlay individual donor data points as a jittered stripplot
sns.stripplot(
    data=plot_df, x='log2fc', y='Label', order=label_order,
    color='black', alpha=0.9, size=3.5, jitter=0.2, ax=ax
)

# Chart styling and layout adjustments
ax.set_box_aspect(1)
ax.axvline(0, color='black', linestyle='--', alpha=0.7, linewidth=2)
sns.despine(ax=ax, top=True, right=True)
ax.xaxis.grid(True, linestyle='--', alpha=0.4)

ax.set_title('IQGAP2', fontsize=FONT_BIG, fontweight='bold', fontstyle='italic', pad=25)
ax.set_xlabel('Log2 Fold Change\n(per donor: macrophages vs microglia)',
             fontsize=FONT_SMALL, fontweight='bold', labelpad=15)
ax.set_ylabel('')
ax.tick_params(axis='both', labelsize=FONT_SMALL)

plt.tight_layout()
plt.show()

In [ ]:
# 1. Group datasets by evolutionary origin
human_dfs = [result_df_1, result_df_2, result_df_3, result_df_4, 
             result_df_5, result_df_8, result_df_9]
nhp_dfs = [result_df_6, result_df_7]

# 2. Concatenate dataframes and assign dataset IDs and species labels
df_human = pd.concat([df.assign(Dataset=f'DS{i}') for df, i in zip(human_dfs, [1,2,3,4,5,8,9])], ignore_index=True)
df_human['Species'] = 'Human'

df_nhp = pd.concat([df.assign(Dataset=f'DS{i}') for df, i in zip(nhp_dfs, [6,7])], ignore_index=True)
df_nhp['Species'] = 'NHP'

plot_df = pd.concat([df_human, df_nhp], ignore_index=True)

# 3. Configure distinct markers for each dataset and global colors
markers = {'DS1':'o', 'DS2':'s', 'DS3':'^', 'DS4':'v', 'DS5':'D', 'DS6':'*', 'DS7':'p', 'DS8':'h', 'DS9':'X'}
colors_map = {'Human':'skyblue', 'NHP':'coral'}

# Fix seed to ensure reproducible jitter layout across multiple runs
np.random.seed(42)

# 4. Generate the visualization plot
fig, ax = plt.subplots(figsize=(8, 6), dpi=300)
sns.set_style("white")

# 🟦 Background layers: Boxplots (ZORDER=2: middle ground)
sns.boxplot(
    data=plot_df, x='Species', y='log2fc',
    palette=[colors_map['Human'], colors_map['NHP']],
    width=0.5, showfliers=False, linewidth=1.5, linecolor='black', 
    ax=ax, zorder=2
)

# Foreground layers: Individual donor data points with jitter (ZORDER=3)
for ds in plot_df['Dataset'].unique():
    sub = plot_df[plot_df['Dataset'] == ds]
    sp = sub['Species'].iloc[0]
    
    # Define baseline X position: 0 for Human, 1 for NHP
    x_val = 0 if sp == 'Human' else 1
    # Add a random uniform horizontal noise (jitter)
    jitter = np.random.uniform(-0.18, 0.18, size=len(sub))
    
    ax.scatter(x_val + jitter, sub['log2fc'], 
               marker=markers.get(ds, 'o'), 
               c=colors_map[sp], s=50, alpha=0.85, 
               edgecolor='black', linewidth=0.3, label=f"{ds}", zorder=3)

# Chart styling, gridlines and aesthetics
ax.axhline(0, color='black', linestyle='--', alpha=0.7, linewidth=2, zorder=1)
sns.despine(ax=ax, top=True, right=True)
ax.yaxis.grid(True, linestyle='--', alpha=0.4)

ax.set_title('IQGAP2', fontsize=23, fontweight='bold', fontstyle='italic', pad=20)
ax.set_ylabel('Log2 Fold Change', fontsize=18, fontweight='bold')
ax.set_xlabel('')
ax.tick_params(axis='both', labelsize=18)
ax.set_xticklabels([f'Human\n(n={len(df_human)})', f'NHP\n(n={len(df_nhp)})'])

# Setup layout legend positioning
ax.legend(title='Dataset', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=12, title_fontsize=14)
plt.tight_layout()
plt.show()